# 14 - Pemeringkatan Influencer: Entropy Weight + EDAS

Tahap akhir: menggabungkan skor sentimen (tahap 13) dengan metrik akun (followers, shares,
engagement) jadi satu matriks keputusan, hitung bobot kriteria otomatis (Entropy Weight), lalu
rangking influencer pakai EDAS (Evaluation based on Distance from Average Solution).

**Kriteria yang dipakai:** **C1** (sentimen), **C2** (followers), **C4** (rata-rata shares), **C5**
(engagement rate = rata-rata total interaksi / followers). **C3 (views) di-drop** karena data
views tidak tersimpan lengkap pas scraping -- ini keterbatasan metodologi yang perlu disebut di
skripsi.

**Skenario yang dibandingkan (3 x 3 = 9 kombinasi):**
- C1 dicoba 3 varian: `3class` (positif=+1/netral=0/negatif=-1), `3class_hanya_positif`
  (positif=+1/netral=0/negatif=0), `binary` (varian B: negatif=-1/non-negatif=+1).
- Threshold minimal komentar bersih per influencer dicoba 3 skenario: **30, 50, 100**.
- Filter tambahan: **followers >= 10.000** (batasan masalah proposal) dan **minimal 10 video**
  per influencer -- biar C4/C5 (rata-rata per video) tidak rawan di-drag 1 video viral jadi
  outlier (lihat diagnosa `farhanzubedi` di bagian paling bawah).

**Alur (1 cell per tahap):**
- **Blok A** -- Setup, config, load skor sentimen (tahap 13) + metadata profil & video (tahap 04).
- **Blok B** -- Hitung C2 (followers), C4 (rata-rata shares), C5 (engagement rate) per influencer.
- **Blok C** -- Susun matriks keputusan (gabung C1+C2+C4+C5) + filter followers/video/komentar.
- **Blok D** -- Fungsi Entropy Weight.
- **Blok E** -- Fungsi EDAS.
- **Blok F** -- Jalankan semua 9 kombinasi + simpan + bandingkan stabilitas rangking (Spearman) +
  diagnosa kontribusi kriteria.

**Konsolidasi dari:** `03_Indobert/04_edas_pemeringkatan.ipynb`

**Catatan:** ringan (pandas + numpy saja, tanpa scraping/LLM/GPU). Skip otomatis kalau
`ringkasan_semua_kombinasi.csv` sudah ada.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# -------------------------------------------------------------------------
# KONFIGURASI
# -------------------------------------------------------------------------
FINAL_OUTPUT_ROOT = Path.cwd() / "output"
OUTPUT_DIR = FINAL_OUTPUT_ROOT / "14_edas_pemeringkatan"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PATH_SKOR_INFLUENCER = FINAL_OUTPUT_ROOT / "13_scoring_komentar" / "skor_influencer.csv"
PATH_PROFIL = FINAL_OUTPUT_ROOT / "04_scraping_metadata_konten" / "metadata_profil_influencer.csv"
PATH_VIDEO = FINAL_OUTPUT_ROOT / "04_scraping_metadata_konten" / "metadata_video_konten.csv"

EXCLUDE_USERNAMES = ["drrichardlee"]  # konsisten sama tahap scoring sebelumnya

# --- 3 skema C1 yang mau dicoba ---
SKEMA_C1 = {
    "3class": "C1_3class",
    "3class_hanya_positif": "C1_3class_hanya_positif",
    "binary": "C1_binary_varianB",
}

# --- 3 threshold minimal komentar bersih per influencer yang mau dicoba ---
DAFTAR_THRESHOLD_KOMENTAR = [30, 50, 100]

# --- Batasan dari proposal (bab 1.3): influencer minimal 10.000 followers ---
MIN_FOLLOWERS = 10_000

# --- Filter tambahan: minimal 10 video, biar C4/C5 (dihitung dari rata-rata per video) ---
# --- ga rawan di-drag outlier dari 1-2 video viral doang ---
MIN_VIDEO = 10

print("Config siap.")
print("Skema C1:", list(SKEMA_C1.keys()))
print("Threshold komentar yg dicoba:", DAFTAR_THRESHOLD_KOMENTAR)
print("Min followers:", MIN_FOLLOWERS)
print("Min jumlah video:", MIN_VIDEO)

## BLOK A - Load 3 sumber data

In [ ]:
skor_influencer = pd.read_csv(PATH_SKOR_INFLUENCER, dtype=str, keep_default_na=False, encoding="utf-8")
# kolom numerik di skor_influencer perlu di-convert balik (tadinya disimpen sbg string biar aman)
for kol in skor_influencer.columns:
    if kol != "username":
        skor_influencer[kol] = pd.to_numeric(skor_influencer[kol], errors="coerce")

df_profil = pd.read_csv(PATH_PROFIL, dtype=str, keep_default_na=False, encoding="utf-8")
df_video = pd.read_csv(PATH_VIDEO, dtype=str, keep_default_na=False, encoding="utf-8")

for nama, df, req in [
    ("skor_influencer.csv", skor_influencer, {"username"} | set(SKEMA_C1.values())),
    ("metadata_profil_influencer.csv", df_profil, {"username", "followers", "scraped_at"}),
    ("metadata_video_konten.csv", df_video, {"username", "video_id", "like_count", "comment_count", "share_count", "save_count"}),
]:
    missing = req - set(df.columns)
    if missing:
        raise ValueError(f"{nama} kurang kolom: {missing}")

print(f"skor_influencer : {len(skor_influencer):,} influencer")
print(f"metadata_profil : {len(df_profil):,} baris")
print(f"metadata_video  : {len(df_video):,} baris")

## BLOK B - Hitung C2 (followers), C4 (rata-rata shares), C5 (engagement rate)

In [ ]:
def ambil_followers(df_profil, exclude_usernames):
    df = df_profil[~df_profil["username"].isin(exclude_usernames)].copy()
    df["followers"] = pd.to_numeric(df["followers"], errors="coerce")
    # kalau 1 username muncul >1 baris (di-scrape ulang beberapa kali), ambil yg paling baru
    df = df.sort_values("scraped_at").drop_duplicates(subset="username", keep="last")
    return df[["username", "followers"]].reset_index(drop=True)


followers_df = ambil_followers(df_profil, EXCLUDE_USERNAMES)
print(f"Followers ter-load: {len(followers_df)} influencer")
print(followers_df.sort_values("followers", ascending=False).head(10).to_string())

In [ ]:
def agregasi_video(df_video, exclude_usernames):
    df = df_video[~df_video["username"].isin(exclude_usernames)].copy()
    for kol in ["like_count", "comment_count", "share_count", "save_count"]:
        df[kol] = pd.to_numeric(df[kol], errors="coerce").fillna(0)
    df["total_interaksi"] = df["like_count"] + df["comment_count"] + df["share_count"] + df["save_count"]
    agg = df.groupby("username").agg(
        C4_rata_shares=("share_count", "mean"),
        rata_total_interaksi=("total_interaksi", "mean"),
        jumlah_video=("video_id", "nunique"),
    ).reset_index()
    return agg


video_agg = agregasi_video(df_video, EXCLUDE_USERNAMES)
print(f"Video agg ter-load: {len(video_agg)} influencer")
print(video_agg.sort_values("C4_rata_shares", ascending=False).head(10).to_string())

In [ ]:
metrik_akun = followers_df.merge(video_agg, on="username", how="inner")
metrik_akun["C5_engagement_rate"] = metrik_akun["rata_total_interaksi"] / metrik_akun["followers"]

print(f"Metrik akun lengkap: {len(metrik_akun)} influencer "
      f"(irisan yg ada DI followers.csv MAUPUN video.csv)")
print(metrik_akun.sort_values("C5_engagement_rate", ascending=False).head(10).to_string())

## BLOK C - Susun Matriks Keputusan + Filter

In [ ]:
def susun_matriks(skor_influencer_df, nama_skema, metrik_akun_df,
                   min_followers=MIN_FOLLOWERS, min_komentar=30, min_video=MIN_VIDEO):
    """Gabungin C1 (sesuai skema) + C2/C4/C5 dari metrik_akun, lalu filter followers, jumlah
    video, & jumlah komentar. Return: df matriks (index=username, kolom C1/C2/C4/C5) + jumlah
    yg kefilter.

    Filter jumlah video penting karena C4/C5 dihitung sbg RATA-RATA per video -- kalau cuma
    dihitung dari sedikit video, rata-ratanya rawan di-drag jauh sama 1 video yg kebetulan
    viral (lihat kasus farhanzubedi di bagian diagnosa paling bawah)."""
    kolom_c1 = SKEMA_C1[nama_skema]
    kolom_p = "P_" + kolom_c1.replace("C1_", "")

    df = skor_influencer_df[["username", kolom_c1, kolom_p]].merge(
        metrik_akun_df, on="username", how="inner")

    sebelum = len(df)
    df = df[df["followers"] >= min_followers]
    df = df[df["jumlah_video"] >= min_video]
    df = df[df[kolom_p] >= min_komentar]
    sesudah = len(df)

    matriks = df.rename(columns={
        kolom_c1: "C1", "followers": "C2", "C4_rata_shares": "C4", "C5_engagement_rate": "C5",
    }).set_index("username")[["C1", "C2", "C4", "C5"]]

    return matriks, sebelum, sesudah


print("Fungsi susun_matriks() siap.")

### Coba 1 kombinasi dulu (skema 3class, threshold 30)

In [ ]:
matriks_contoh, sebelum, sesudah = susun_matriks(
    skor_influencer, "3class", metrik_akun,
    min_followers=MIN_FOLLOWERS, min_komentar=30, min_video=MIN_VIDEO)
print(f"Influencer sebelum filter: {sebelum}, sesudah filter (followers>={MIN_FOLLOWERS}, "
      f"video>={MIN_VIDEO}, komentar>=30): {sesudah}")
print(matriks_contoh.sort_values("C1", ascending=False).to_string())

## BLOK D - Entropy Weight

In [ ]:
def hitung_bobot_entropy(matriks, kolom_kriteria):
    """Normalisasi probabilitas pij, entropy Ej, derajat diversifikasi dj, bobot wj.
    Kalau pij=0, pij*ln(pij) dianggap 0."""
    m = len(matriks)
    k = 1 / np.log(m)

    detail = []
    for kol in kolom_kriteria:
        x = matriks[kol].values.astype(float)
        p = x / x.sum()
        p_lnp = np.where(p > 0, p * np.log(p), 0.0)
        Ej = -k * p_lnp.sum()
        dj = 1 - Ej
        detail.append({"kriteria": kol, "Ej": Ej, "dj": dj})

    total_d = sum(d["dj"] for d in detail)
    bobot = {}
    for d in detail:
        d["wj"] = d["dj"] / total_d
        bobot[d["kriteria"]] = d["wj"]

    return bobot, pd.DataFrame(detail).set_index("kriteria")


print("Fungsi hitung_bobot_entropy() siap.")

bobot_contoh, detail_entropy_contoh = hitung_bobot_entropy(matriks_contoh, ["C1", "C2", "C4", "C5"])
print("\nBobot entropy (contoh, skema 3class, threshold 30):")
print(detail_entropy_contoh.to_string())
print("\nTotal bobot (harus = 1.0):", sum(bobot_contoh.values()))

## BLOK E - EDAS (Evaluation based on Distance from Average Solution)

In [ ]:
def hitung_edas(matriks, kolom_kriteria, bobot):
    """Semua kriteria di penelitian ini tipe BENEFIT, jadi cuma pakai rumus PDA/NDA versi
    benefit."""
    df = matriks.copy()
    AV = {kol: df[kol].mean() for kol in kolom_kriteria}

    for kol in kolom_kriteria:
        avj = AV[kol]
        df[f"PDA_{kol}"] = np.maximum(0, df[kol] - avj) / avj
        df[f"NDA_{kol}"] = np.maximum(0, avj - df[kol]) / avj

    df["SP"] = sum(bobot[kol] * df[f"PDA_{kol}"] for kol in kolom_kriteria)
    df["SN"] = sum(bobot[kol] * df[f"NDA_{kol}"] for kol in kolom_kriteria)

    max_SP, max_SN = df["SP"].max(), df["SN"].max()
    df["NSP"] = df["SP"] / max_SP if max_SP != 0 else 0.0
    df["NSN"] = 1 - (df["SN"] / max_SN if max_SN != 0 else 0.0)

    df["AS"] = 0.5 * (df["NSP"] + df["NSN"])
    df["rank"] = df["AS"].rank(ascending=False, method="min").astype(int)
    return df.sort_values("AS", ascending=False)


print("Fungsi hitung_edas() siap.")

hasil_contoh = hitung_edas(matriks_contoh, ["C1", "C2", "C4", "C5"], bobot_contoh)
print("\nHasil EDAS (contoh, skema 3class, threshold 30), top 10:")
print(hasil_contoh[["C1", "C2", "C4", "C5", "SP", "SN", "AS", "rank"]].head(10).to_string())

## BLOK F - Jalankan Semua Kombinasi (3 skema x 3 threshold) + Bandingkan

In [ ]:
RINGKASAN_CSV = OUTPUT_DIR / "ringkasan_semua_kombinasi.csv"

if RINGKASAN_CSV.exists():
    print(f"[EDAS] {RINGKASAN_CSV} sudah ada -> skip hitung ulang semua kombinasi (baca file yg ada).")
    ringkasan_df = pd.read_csv(RINGKASAN_CSV)
    semua_hasil = {}
    for _, row in ringkasan_df.iterrows():
        key = f"{row['skema']}__min{row['threshold_komentar']}"
        semua_hasil[key] = {"hasil": pd.read_csv(OUTPUT_DIR / f"edas_{key}.csv", index_col="username")}
    print(ringkasan_df.to_string())
else:
    semua_hasil = {}
    ringkasan_kombinasi = []

    for nama_skema in SKEMA_C1:
        for threshold in DAFTAR_THRESHOLD_KOMENTAR:
            matriks, sebelum, sesudah = susun_matriks(
                skor_influencer, nama_skema, metrik_akun,
                min_followers=MIN_FOLLOWERS, min_komentar=threshold, min_video=MIN_VIDEO,
            )
            bobot, detail_entropy = hitung_bobot_entropy(matriks, ["C1", "C2", "C4", "C5"])
            hasil = hitung_edas(matriks, ["C1", "C2", "C4", "C5"], bobot)

            key = f"{nama_skema}__min{threshold}"
            semua_hasil[key] = {"matriks": matriks, "bobot": bobot, "detail_entropy": detail_entropy, "hasil": hasil}
            ringkasan_kombinasi.append({
                "skema": nama_skema, "threshold_komentar": threshold,
                "jumlah_influencer": sesudah, "top1": hasil.index[0], "top1_AS": hasil.iloc[0]["AS"],
            })
            print(f'[{key}] {sesudah} influencer -- top1: {hasil.index[0]} (AS={hasil.iloc[0]["AS"]:.3f})')

    ringkasan_df = pd.DataFrame(ringkasan_kombinasi)
    print("\nRingkasan semua kombinasi:")
    print(ringkasan_df.to_string())

    for key, data in semua_hasil.items():
        data["hasil"].to_csv(OUTPUT_DIR / f"edas_{key}.csv", encoding="utf-8")
    ringkasan_df.to_csv(RINGKASAN_CSV, index=False, encoding="utf-8")
    print(f"\nTersimpan {len(semua_hasil)} file edas_<skema>__min<threshold>.csv ke {OUTPUT_DIR}")
    print("Plus ringkasan_semua_kombinasi.csv")

### Tampilkan top-10 tiap kombinasi

In [ ]:
for key, data in semua_hasil.items():
    print(f"\n=== {key} ===")
    print(data["hasil"][["C1", "C2", "C4", "C5", "AS", "rank"]].head(10).to_string())

### Bandingkan stabilitas rangking antar kombinasi (Spearman)

In [ ]:
tabel_rank = None
for key, data in semua_hasil.items():
    kolom_rank = data["hasil"][["rank"]].rename(columns={"rank": key})
    tabel_rank = kolom_rank if tabel_rank is None else tabel_rank.join(kolom_rank, how="outer")

print("Tabel rank tiap influencer, per kombinasi (NaN = influencer kefilter di kombinasi itu):")
print(tabel_rank.to_string())

print("\nKorelasi rank Spearman antar kombinasi (makin deket 1.0 = makin konsisten/stabil):")
korelasi_rank = tabel_rank.corr(method="spearman")
print(korelasi_rank.round(3).to_string())

### Diagnosa: kenapa akun yang postingannya dikit bisa rank 1?

Dipicu pertanyaan: C1 (sentimen) `farhanzubedi` di beberapa skema malah mendekati 0/negatif,
tapi dia tetap rank 1 di semua kombinasi. Cell di bawah nampilin tabel bobot entropy ($w_j$)
tiap kombinasi -- biar ketahuan kriteria mana yang paling "berbobot" di tiap skenario (entropy
ngasih bobot besar ke kriteria yang datanya paling tersebar/variatif, jadi kalau C4 atau C5
datanya tersebar jauh karena ada outlier, bobotnya bisa jadi gede dan outlier itu jadi dominan
ke skor akhir), lalu bedah kontribusi tiap kriteria ke SP/SN-nya `farhanzubedi` secara spesifik.

In [ ]:
if all("bobot" in d for d in semua_hasil.values()):
    tabel_bobot = pd.DataFrame({key: data["bobot"] for key, data in semua_hasil.items()}).T
    tabel_bobot.index.name = "kombinasi"
    print("Bobot entropy (wj) per kriteria, tiap kombinasi:")
    print(tabel_bobot.round(4).to_string())

    print("\nKriteria dengan bobot terbesar per kombinasi:")
    for key in tabel_bobot.index:
        kriteria_max = tabel_bobot.loc[key].idxmax()
        print(f"  {key:32s}: {kriteria_max} (wj={tabel_bobot.loc[key, kriteria_max]:.4f})")
else:
    print("[Bobot entropy] Dibaca dari file CSV yang sudah ada -- detail bobot per kriteria tidak "
          "ikut tersimpan di edas_<key>.csv (cuma hasil akhir). Hapus ringkasan_semua_kombinasi.csv "
          "lalu jalankan ulang cell BLOK F kalau butuh detail bobot ini lagi.")

In [ ]:
def bedah_kontribusi(key, username, kolom_kriteria=["C1", "C2", "C4", "C5"]):
    """Nunjukin nilai mentah, PDA, NDA, dan sumbangan ke SP (bobot*PDA) tiap kriteria buat
    1 username di 1 kombinasi -- buat nunjukin kriteria mana yg paling 'ngangkat' dia."""
    data = semua_hasil[key]
    hasil = data["hasil"]
    bobot = data.get("bobot")
    if bobot is None:
        print(f"[{key}] Bobot entropy tidak tersedia (hasil dibaca dari CSV lama, tanpa detail bobot).")
        return None
    if username not in hasil.index:
        print(f"{username} tidak ada di kombinasi {key} (mungkin kefilter).")
        return None
    baris = hasil.loc[username]
    rincian = []
    for kol in kolom_kriteria:
        rincian.append({
            "kriteria": kol,
            "nilai": baris[kol],
            "wj": bobot[kol],
            "PDA": baris[f"PDA_{kol}"],
            "NDA": baris[f"NDA_{kol}"],
            "sumbangan_SP": bobot[kol] * baris[f"PDA_{kol}"],
            "sumbangan_SN": bobot[kol] * baris[f"NDA_{kol}"],
        })
    rincian_df = pd.DataFrame(rincian).set_index("kriteria")
    print(f'[{key}] {username} -- AS={baris["AS"]:.4f}, rank={int(baris["rank"])}')
    print(rincian_df.round(4).to_string())
    print(f'Total SP={baris["SP"]:.4f}  (kriteria penyumbang SP terbesar: '
          f'{rincian_df["sumbangan_SP"].idxmax()})')
    return rincian_df


_ = bedah_kontribusi("3class__min30", "farhanzubedi")